# Crawl dữ liệu cổ phiếu Việt Nam từ CafeF

**Lấy dữ liệu từ CafeF**

1. Mở [CafeF](https://cafef.vn/), tìm mã cổ phiếu cần tra và vào mục lịch sử giá/giao dịch để xem dữ liệu ngày có sẵn.
2. Trong notebook này, dữ liệu được lấy từ [CafeF PriceHistory API](https://cafef.vn/du-lieu/Ajax/PageNew/DataHistory/PriceHistory.ashx). 
3. Mở trang lịch sử giá CafeF trên trình duyệt, nhấn `F12` → `Network` → `Fetch/XHR`, đổi mã hoặc khoảng ngày và tìm request `PriceHistory.ashx`.

Trước khi crawl, tự tạo file `tickers.csv` trong thư mục workspace `Stock_price`. 


In [4]:
import logging
import time
from pathlib import Path

import pandas as pd
import requests

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("crawl_cafef")

BASE_DIR = Path.cwd()

API_URL = "https://cafef.vn/du-lieu/Ajax/PageNew/DataHistory/PriceHistory.ashx"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/128.0.0.0 Safari/537.36",
    "Referer": "https://cafef.vn/",
}

TICKERS_FILE = BASE_DIR / "tickers.csv"
START_DATE = "01/01/2026"
END_DATE = "28/01/2026"
PAGE_SIZE = 30
SLEEP_SECONDS = 1.0
OUTPUT_CSV = BASE_DIR / "LichSuGia_Crawled.csv"

In [ ]:
# Tạo tickers.csv thủ công trong thư mục workspace.

In [3]:
# Mapping field JSON trả về từ CafeF -> tên cột tiếng Việt, GIỮ NGUYÊN như file gốc
# để không phải sửa bước Transform trong ETL_Pipeline.ipynb
FIELD_MAP = {
    "Ngay": "Ngày",
    "GiaDongCua": "Giá đóng cửa",
    "GiaDieuChinh": "Giá điều chỉnh",
    "ThayDoi": "Thay đổi",
    "KhoiLuongKhopLenh": "Khối lượng khớp lệnh",
    "GiaTriKhopLenh": "Giá trị khớp lệnh",
    "KLThoaThuan": "Khối lượng thỏa thuận",
    "GtThoaThuan": "Giá trị thỏa thuận",
    "KhoiLuongThoaThuan": "Khối lượng thỏa thuận",
    "GiaTriThoaThuan": "Giá trị thỏa thuận",
    "GiaMoCua": "Giá mở cửa",
    "GiaCaoNhat": "Giá cao nhất",
    "GiaThapNhat": "Giá thấp nhất",
}


In [2]:
def load_tickers(path: Path) -> list[str]:
    """Đọc danh sách mã từ tickers.csv do người dùng tự tạo."""
    if not path.exists():
        raise FileNotFoundError(
            f"Không tìm thấy {path}. Hãy tạo CSV có một cột tên Ticker, mỗi mã một dòng."
        )

    df = pd.read_csv(path, encoding="utf-8-sig")
    if "Ticker" not in df.columns:
        raise ValueError("tickers.csv cần có cột tên 'Ticker'.")

    tickers = (
        df["Ticker"].dropna().astype(str).str.strip().str.upper()
    )
    tickers = tickers[tickers != ""].drop_duplicates().tolist()
    if not tickers:
        raise ValueError("tickers.csv chưa có mã cổ phiếu nào.")

    return tickers


def get_price_history(symbol: str, start_date: str, end_date: str, page_size: int = 30) -> pd.DataFrame:
    """Gọi API CafeF lấy lịch sử giá 1 mã, trả về DataFrame (cột tiếng Việt như file gốc)."""
    params = {
        "Symbol": symbol,
        "StartDate": start_date,
        "EndDate": end_date,
        "PageIndex": 1,
        "PageSize": page_size,
    }
    resp = requests.get(API_URL, params=params, headers=HEADERS, timeout=15)
    resp.raise_for_status()
    payload = resp.json()

    if not payload.get("Success", False):
        log.warning(f"[{symbol}] API trả về Success=False: {payload.get('Message')}")
        return pd.DataFrame()

    response_data = payload.get("Data", {})
    records = response_data.get("Data", []) if isinstance(response_data, dict) else []
    if not records:
        log.warning(f"[{symbol}] Không có dữ liệu trong khoảng ngày đã chọn.")
        return pd.DataFrame()

    df = pd.DataFrame(records)
    df = df.rename(columns=FIELD_MAP)
    df.insert(0, "Mã", symbol)
    keep_cols = ["Mã"] + list(FIELD_MAP.values())
    df = df[[c for c in keep_cols if c in df.columns]]
    return df

In [3]:
def crawl_all(tickers_file: str | Path, start_date: str, end_date: str,
               page_size: int, sleep_seconds: float) -> pd.DataFrame:
    tickers = load_tickers(Path(tickers_file))
    log.info(f"Chuẩn bị crawl {len(tickers)} mã, khoảng {start_date} - {end_date}...")

    all_frames = []
    for i, symbol in enumerate(tickers, start=1):
        try:
            df_symbol = get_price_history(symbol, start_date, end_date, page_size)
            if not df_symbol.empty:
                all_frames.append(df_symbol)
            log.info(f"[{i}/{len(tickers)}] {symbol}: {len(df_symbol)} dòng")
        except Exception as e:
            log.error(f"[{symbol}] Lỗi khi crawl: {e}")
        time.sleep(sleep_seconds)

    if not all_frames:
        raise RuntimeError("Không crawl được dữ liệu cho bất kỳ mã nào - kiểm tra lại API_URL/HEADERS.")

    df_all = pd.concat(all_frames, ignore_index=True)
    log.info(f"Tổng cộng crawl được {len(df_all):,} dòng cho {df_all['Mã'].nunique()} mã.")
    return df_all

In [ ]:
if __name__ == "__main__":
    df_result = crawl_all(TICKERS_FILE, START_DATE, END_DATE, PAGE_SIZE, SLEEP_SECONDS)
    df_result.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    log.info(f"Đã lưu kết quả crawl ra: {OUTPUT_CSV}")
    log.info("Dùng LichSuGia_Crawled.csv làm đầu vào cho ETL_Pipeline.ipynb.")

13:03:06 | INFO | Chuẩn bị crawl 403 mã, khoảng 01/01/2026 - 28/01/2026...
13:03:07 | INFO | [1/403] AAA: 20 dòng
13:03:09 | INFO | [2/403] AAM: 20 dòng
13:03:10 | INFO | [3/403] AAT: 20 dòng
13:03:12 | INFO | [4/403] ABR: 20 dòng
13:03:14 | INFO | [5/403] ABS: 20 dòng
13:03:15 | INFO | [6/403] ABT: 20 dòng
13:03:17 | INFO | [7/403] ACB: 20 dòng
13:03:19 | INFO | [8/403] ACC: 20 dòng
13:03:20 | INFO | [9/403] ACG: 20 dòng
13:03:22 | INFO | [10/403] ACL: 20 dòng
13:03:24 | INFO | [11/403] ADG: 20 dòng
13:03:25 | INFO | [12/403] ADP: 20 dòng
13:03:27 | INFO | [13/403] ADS: 20 dòng
13:03:29 | INFO | [14/403] AFX: 20 dòng
13:03:30 | INFO | [15/403] AGG: 20 dòng
13:03:32 | INFO | [16/403] AGR: 20 dòng
13:03:34 | INFO | [17/403] ANT: 20 dòng
13:03:36 | INFO | [18/403] ANV: 20 dòng
13:03:37 | INFO | [19/403] APG: 20 dòng
13:03:38 | INFO | [20/403] APH: 20 dòng
13:03:40 | INFO | [21/403] ASG: 20 dòng
13:03:42 | INFO | [22/403] ASM: 20 dòng
13:03:43 | INFO | [23/403] ASP: 20 dòng
13:03:45 | INF